# Notebook 3 – Missing Value Handling
**File:** `03_Missing_Value_Handling.ipynb`

## Learning objectives
By the end, you will be able to identify missing data, understand why it is missing, compare imputation methods, and choose a method based on the data and business problem.

> Important: Do not automatically replace every missing value with the mean. First investigate why values are missing, how much data is missing, and whether the missingness itself carries information.

## 1. What are missing values?
A missing value means that a field has no recorded value. In pandas, missing values are commonly represented by `NaN`, `None`, or `NaT`.

**Why handling is required:** Many analyses and machine-learning algorithms cannot work correctly with missing values. Missingness can also bias statistics and model predictions if it is handled carelessly.

**Simple example:** A student record has no value for `Age`.

**Real-world example:** A hospital patient record has no recorded blood-pressure reading.

**Business example:** An online order has no delivery date because it has not shipped yet.

**AI/ML use case:** A house-price model may need a strategy for missing `LotArea` or `GarageArea` values. The correct strategy depends on why those values are absent.

In [2]:
import numpy as np
import pandas as pd

df = pd.DataFrame({
    "Student": ["Asha", "Ravi", "Mina", "John", "Sara"],
    "Age": [20, np.nan, 22, 21, np.nan],
    "Marks": [85, 78, np.nan, 91, 88],
    "City": ["Hyderabad", "Chennai", None, "Hyderabad", "Delhi"]
})
display(df)

,Student,Age,Marks,City
0,Asha,20.0,85.0,Hyderabad
1,Ravi,NaN,78.0,Chennai
2,Mina,22.0,NaN,NaN
3,John,21.0,91.0,Hyderabad
4,Sara,NaN,88.0,Delhi


## 2. Why do missing values occur?
Common causes include:
- A person skipped a question.
- A sensor or system failed to collect a measurement.
- Data was lost during transfer or integration.
- A field was not applicable (for example, `SpouseName` for an unmarried person).
- A value is intentionally hidden or not yet available.

Before imputing, distinguish **unknown**, **not applicable**, **not yet available**, and **not collected**. These meanings may need different treatment.

## 3. MCAR, MAR, and MNAR

| Type | Meaning | Example | Implication |
|---|---|---|---|
| MCAR | Missing Completely At Random. Missingness is unrelated to observed or unobserved data. | A few forms are randomly damaged. | Complete-case analysis may be less biased, but still loses data. |
| MAR | Missing At Random conditional on observed variables. | Older customers are less likely to answer an optional income question; age is observed. | Use observed variables such as age in a justified imputation model. |
| MNAR | Missing Not At Random. Missingness depends on the missing value itself or an unobserved factor. | People with very high income are less likely to report income. | Simple imputation may be biased; investigate, model missingness, and use domain knowledge. |

The labels describe the mechanism, not the missing-value percentage. A dataset with 2% missing data can still be MNAR.

## 4. Detecting missing values and calculating percentages

In [3]:
# Count missing values in each column
print(df.isna().sum())

# Percentage missing in each column
missing_pct = df.isna().mean().mul(100).round(2)
print(missing_pct)

# Rows containing at least one missing value
display(df[df.isna().any(axis=1)])

# Visual-style summary without an extra plotting library
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": df.isna().mean().mul(100).round(2)
})
display(missing_summary)

Student    0
Age        2
Marks      1
City       1
dtype: int64
Student     0.0
Age        40.0
Marks      20.0
City       20.0
dtype: float64


,Student,Age,Marks,City
1,Ravi,NaN,78.0,Chennai
2,Mina,22.0,NaN,NaN
4,Sara,NaN,88.0,Delhi


,missing_count,missing_percent
Student,0,0.0
Age,2,40.0
Marks,1,20.0
City,1,20.0


**When to use:** Run these checks at the start of EDA and again after joins or transformations.

**When not to use:** A missingness count alone is not enough to decide how to impute. It does not explain the reason or pattern.

**Advantages:** Fast, clear, and helps prioritize columns.

**Limitations:** Does not identify MCAR/MAR/MNAR by itself and does not tell whether a missing value is meaningful.

## 5. Dropping rows
Remove rows only when the loss is acceptable and the remaining records are sufficiently representative.

In [4]:
rows_dropped = df.dropna()
display(rows_dropped)
print("Rows before:", len(df), "| Rows after:", len(rows_dropped))

,Student,Age,Marks,City
0,Asha,20.0,85.0,Hyderabad
3,John,21.0,91.0,Hyderabad


Rows before: 5 | Rows after: 2


- **Use when:** Only a small number of rows are incomplete, the missingness is plausibly MCAR, and enough representative data remains.
- **Do not use when:** Missingness is substantial, important groups are disproportionately removed, or the dataset is already small.
- **Advantages:** Simple; no artificial values are introduced.
- **Limitations:** Reduces sample size and can introduce selection bias.

## 6. Dropping columns

In [5]:
# Work on a copy. Here City is only an illustration; do not drop it without a reason.
df_drop_column = df.drop(columns=["City"])
display(df_drop_column)

,Student,Age,Marks
0,Asha,20.0,85.0
1,Ravi,NaN,78.0
2,Mina,22.0,NaN
3,John,21.0,91.0
4,Sara,NaN,88.0


- **Use when:** A feature has an extremely high missing rate and is not essential, or it cannot be recovered reliably.
- **Do not use when:** The feature is important to the business question or missingness itself is informative.
- **Advantages:** Simplifies the dataset and avoids unreliable imputation.
- **Limitations:** Removes potentially useful signal; a high missing percentage alone is not sufficient justification.

## 7. Mean imputation
Mean replaces missing numerical values with the arithmetic average. It is sensitive to outliers and can reduce variance.

In [6]:
mean_value = df["Marks"].mean()
df_mean = df.copy()
df_mean["Marks"] = df_mean["Marks"].fillna(mean_value)
print("Mean used:", mean_value)
display(df_mean)

Mean used: 85.5


,Student,Age,Marks,City
0,Asha,20.0,85.0,Hyderabad
1,Ravi,NaN,78.0,Chennai
2,Mina,22.0,85.5,NaN
3,John,21.0,91.0,Hyderabad
4,Sara,NaN,88.0,Delhi


- **Use when:** A numerical feature is reasonably symmetric, has few missing values, and the mean is meaningful.
- **Do not use when:** The feature is highly skewed, contains influential outliers, or missingness is MNAR.
- **Advantages:** Easy and preserves row count.
- **Limitations:** Can distort the distribution, reduce variance, and weaken relationships between variables.

## 8. Median imputation
The median is the middle value and is more robust to extreme values than the mean.

In [7]:
median_value = df["Marks"].median()
df_median = df.copy()
df_median["Marks"] = df_median["Marks"].fillna(median_value)
print("Median used:", median_value)
display(df_median)

Median used: 86.5


,Student,Age,Marks,City
0,Asha,20.0,85.0,Hyderabad
1,Ravi,NaN,78.0,Chennai
2,Mina,22.0,86.5,NaN
3,John,21.0,91.0,Hyderabad
4,Sara,NaN,88.0,Delhi


- **Use when:** Numerical data is skewed or contains outliers.
- **Do not use when:** The feature is categorical or when a single median hides important groups.
- **Advantages:** Robust to outliers and simple.
- **Limitations:** Can reduce variability and ignore relationships with other features.

## 9. Mode imputation
The mode is the most frequent value. It is commonly used for categorical features.

In [8]:
mode_value = df["City"].mode(dropna=True).iloc[0]
df_mode = df.copy()
df_mode["City"] = df_mode["City"].fillna(mode_value)
print("Mode used:", mode_value)
display(df_mode)

Mode used: Hyderabad


,Student,Age,Marks,City
0,Asha,20.0,85.0,Hyderabad
1,Ravi,NaN,78.0,Chennai
2,Mina,22.0,NaN,Hyderabad
3,John,21.0,91.0,Hyderabad
4,Sara,NaN,88.0,Delhi


- **Use when:** A categorical column has a small amount of missingness and its most frequent category is a reasonable replacement.
- **Do not use when:** Categories are balanced, missingness is meaningful, or the mode would dominate the data artificially.
- **Advantages:** Easy and keeps values within valid categories.
- **Limitations:** Increases the frequency of the dominant category and may introduce bias.

## 10. Constant-value imputation
Use a meaningful constant such as `"Unknown"` or `0` only when that value has a valid interpretation. For numerical data, a sentinel such as `-1` must be clearly documented and must not be confused with a valid measurement.

In [9]:
df_constant = df.copy()
df_constant["City"] = df_constant["City"].fillna("Unknown")
display(df_constant)

,Student,Age,Marks,City
0,Asha,20.0,85.0,Hyderabad
1,Ravi,NaN,78.0,Chennai
2,Mina,22.0,NaN,Unknown
3,John,21.0,91.0,Hyderabad
4,Sara,NaN,88.0,Delhi


- **Use when:** “Unknown” or “Not provided” is a meaningful category, or a domain-approved sentinel is required.
- **Do not use when:** The constant could be mistaken for a real value (for example, age 0 for adults).
- **Advantages:** Preserves rows and makes missingness explicit.
- **Limitations:** Can create an artificial group or distort numerical patterns.

## 11. Forward fill and backward fill
These methods copy the previous or next observed value. They are mainly suitable for ordered/time-series data where carrying a value is justified.

In [10]:
time_df = pd.DataFrame({
    "Date": pd.date_range("2026-01-01", periods=5, freq="D"),
    "Temperature": [28.0, np.nan, np.nan, 31.0, np.nan]
}).set_index("Date")

display(time_df.ffill().rename(columns={"Temperature": "Forward_fill"}))
display(time_df.bfill().rename(columns={"Temperature": "Backward_fill"}))

,Forward_fill
Date,
2026-01-01,28.0
2026-01-02,28.0
2026-01-03,28.0
2026-01-04,31.0
2026-01-05,31.0


,Backward_fill
Date,
2026-01-01,28.0
2026-01-02,31.0
2026-01-03,31.0
2026-01-04,31.0
2026-01-05,NaN


- **Use when:** Records are correctly sorted and a value is expected to persist until the next measurement (for example, account status).
- **Do not use when:** Values change rapidly, rows are not correctly ordered, or carrying values across customers/groups would be wrong.
- **Advantages:** Preserves temporal continuity and is simple.
- **Limitations:** Can propagate stale or future information. In predictive time-series work, backward fill can cause data leakage.

## 12. Interpolation
Interpolation estimates values between known observations. It is useful for smoothly changing measurements, but the assumption must fit the domain.

In [11]:
interpolation_df = pd.DataFrame({
    "Day": [1, 2, 3, 4, 5],
    "Sales": [100, np.nan, 140, np.nan, 180]
}).set_index("Day")

display(interpolation_df.interpolate(method="linear"))

,Sales
Day,
1,100.0
2,120.0
3,140.0
4,160.0
5,180.0


- **Use when:** Measurements are ordered and a smooth or approximately linear change between observations is plausible.
- **Do not use when:** Data is categorical, highly irregular, or values can jump suddenly.
- **Advantages:** Uses neighboring observations and can preserve trends better than a global constant.
- **Limitations:** Depends on assumptions; edge values may remain missing; can leak future information if used carelessly in forecasting.

## 13. Group-based imputation
Impute within a meaningful group, such as median salary by department. Groups should be sufficiently large and the grouping variable should be available at prediction time.

In [12]:
employees = pd.DataFrame({
    "Department": ["IT", "IT", "HR", "HR", "Sales", "Sales"],
    "Salary": [60000, np.nan, 40000, np.nan, 50000, 55000]
})
employees["Salary_filled"] = employees["Salary"]
employees["Salary_filled"] = employees.groupby("Department")["Salary"].transform(
    lambda s: s.fillna(s.median())
)
display(employees)

,Department,Salary,Salary_filled
0,IT,60000.0,60000.0
1,IT,NaN,60000.0
2,HR,40000.0,40000.0
3,HR,NaN,40000.0
4,Sales,50000.0,50000.0
5,Sales,55000.0,55000.0


- **Use when:** Groups have genuinely different distributions (department, region, product type).
- **Do not use when:** Groups are tiny, unstable, or the group label is unavailable at inference time.
- **Advantages:** Respects group differences better than one global value.
- **Limitations:** Small groups may have no median; grouping can encode bias or create leakage if groups use future information.

## 14. KNN imputation
K-nearest-neighbors imputation finds similar rows and estimates a missing value from their neighbors. Numerical features usually need scaling because distance is scale-sensitive.

In [13]:
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

knn_data = pd.DataFrame({
    "Age": [20, 21, np.nan, 40, 41],
    "Income": [30000, 32000, 31000, np.nan, 60000],
    "Score": [70, 72, 71, 90, np.nan]
})

scaler = StandardScaler()
scaled = scaler.fit_transform(knn_data)
knn_imputer = KNNImputer(n_neighbors=2)
knn_scaled_filled = knn_imputer.fit_transform(scaled)
knn_filled = scaler.inverse_transform(knn_scaled_filled)
display(pd.DataFrame(knn_filled, columns=knn_data.columns).round(2))

,Age,Income,Score
0,20.0,30000.0,70.0
1,21.0,32000.0,72.0
2,20.5,31000.0,71.0
3,40.0,46000.0,90.0
4,41.0,60000.0,81.0


- **Use when:** Features are meaningfully related and similar rows are expected to have similar values.
- **Do not use when:** Dataset is very large, features are mostly unrelated, or distance is not meaningful.
- **Advantages:** Can use relationships among several features.
- **Limitations:** Computationally expensive, sensitive to scaling and feature selection, and may perform poorly with high-dimensional data. Fit it only on training data.

## 15. Iterative imputation
Iterative imputation models each incomplete feature using the other features, repeating the process. It can capture multivariable relationships.

In [14]:
from sklearn.experimental import enable_iterative_imputer  # enables IterativeImputer
from sklearn.impute import IterativeImputer

iterative_data = pd.DataFrame({
    "Age": [20, 21, np.nan, 40, 41, 35],
    "Income": [30000, 32000, 31000, np.nan, 60000, 52000],
    "Score": [70, 72, 71, 90, np.nan, 85]
})

iterative_imputer = IterativeImputer(random_state=42, max_iter=10)
iterative_filled = iterative_imputer.fit_transform(iterative_data)
display(pd.DataFrame(iterative_filled, columns=iterative_data.columns).round(2))

,Age,Income,Score
0,20.00,30000.00,70.00
1,21.00,32000.00,72.00
2,20.48,31000.00,71.00
3,40.00,59007.48,90.00
4,41.00,60000.00,90.57
5,35.00,52000.00,85.00


- **Use when:** Several numerical features are related and a more flexible multivariable approach is justified.
- **Do not use when:** The dataset is too small, relationships are unstable, or the method cannot be validated.
- **Advantages:** Uses relationships among multiple columns and can be more realistic than a single-column statistic.
- **Limitations:** More complex, computationally heavier, model-dependent, and not guaranteed to remove bias—especially under MNAR.

## 16. Prevent data leakage with train/test split
For machine learning, split first, then fit the imputer using training data only. Apply the learned transformation to validation/test data. A pipeline helps enforce this.

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

sample = pd.DataFrame({
    "age": [22, 25, 47, 52, 46, 56, 23, 31, 40, 60, 28, 35],
    "income": [25000, 30000, 60000, 65000, np.nan, 80000, 27000, 42000, 50000, 90000, 32000, np.nan],
    "purchased": [0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0]
})
X = sample[["age", "income"]]
y = sample["purchased"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression())
])
pipeline.fit(X_train, y_train)
print("Test accuracy:", round(pipeline.score(X_test, y_test), 3))

Test accuracy: 1.0


## 17. Choosing a method – quick guide

| Data situation | Possible starting method | Check before choosing |
|---|---|---|
| Few MCAR rows | Drop rows | Sample size and representativeness |
| Skewed numerical feature | Median | Group patterns and outliers |
| Symmetric numerical feature | Mean | Outliers and variance changes |
| Categorical feature | Mode / Unknown | Whether missingness has meaning |
| Ordered time series | Forward fill / interpolation | Time order and leakage |
| Different group distributions | Group median/mode | Group size and availability |
| Related numerical features | KNN / Iterative | Scaling, compute, validation |

## 18. Final checklist
1. Measure missingness by column and by important groups.
2. Ask why values are missing; consider MCAR/MAR/MNAR.
3. Decide whether missingness itself should be represented by an indicator.
4. Choose a method based on data type, domain, and model.
5. Fit imputers on training data only.
6. Compare model performance and inspect distributions before and after.
7. Document the decision and its limitations.